# JSON 검수 및 획 데이터 생성

1. Drive를 연결합니다. 원본 좌표는 `HSQM/dataset/automated_train_dataset/jsons/<폰트>/`에서 가져옵니다.
2. 폰트와 글자를 골라 획을 확인합니다. 기본 동작인 **통과 · 다음**을 누르면 그 JSON을 `ref is true/jsons/<폰트>/`에 복사하고 다음 글자로 넘어갑니다. 같은 폰트의 동명 JSON이 통과 폴더에 있으면 그 사본을 쓰고 검수 화면에서 건너뜁니다.
   - **제외 · 원본 삭제**를 누르면 현재 선택한 JSON을 원본 폴더에서 바로 삭제합니다. 같은 글자의 다른 JSON이 남아 있으면 이어서 확인할 수 있습니다.
3. 마지막 셀에서 통과한 폰트·글자마다 10장을 생성합니다. JSON에서 `THICKNESS_PX` 굵기로 그린 획을 합쳐 target을 만들고, 같은 굵은 획에 가로세로 비율·크기·회전 변형을 적용해 input stroke와 input을 만듭니다. ref 획은 변형 전 JSON에서 `REF_STROKE_THICKNESS_PX` 굵기로 따로 그립니다. 이미 검수했다면 1번 셀과 마지막 셀만 실행하세요.

input stroke를 픽셀별 최댓값으로 합치면 input이 정확히 복원됩니다. 굵은 무변형 target은 NPZ의 `reference`, 얇은 ref 획은 `strokes`, 학습 정답인 input stroke는 `target_masks`에 저장합니다.

출력은 `HSQM/dataset/ref is true/snapshot-…/npz/samples-*.tar`입니다. 학습 노트북의 `DATA_ROOT`에는 생성 셀이 출력한 `snapshot-…` 폴더를 지정하세요. 검수 결과나 설정이 바뀌면 새 폴더에 저장하며, 기존 출력은 보존합니다.


In [1]:
# @title 1. 코드 준비 및 Drive 연결
from pathlib import Path
import subprocess
import sys
import tempfile
from google.colab import drive

root = Path(tempfile.mkdtemp(prefix='hsqm_ref_true_', dir='/content'))
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
                'https://github.com/limhjbrian/hsqm2.git', str(root)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root/'requirements.txt'), 'ipywidgets'], check=True)
sys.path.insert(0, str(root))
for key in tuple(sys.modules):
    if key == 'stroke_model' or key.startswith('stroke_model.'):
        del sys.modules[key]
drive.mount('/content/drive')

from stroke_model.convert_reference_csv import existing_path
from stroke_model.ref_true_dataset import generate_approved, review_coordinates

dataset = existing_path('/content/drive/Shareddrives/2026 자율연구/HSQM/dataset')
json_root = existing_path(dataset/'automated_train_dataset'/'jsons')
destination = dataset/'ref is true'
approved_root = destination/'jsons'
review_path = destination/'json_review.json'


Mounted at /content/drive


In [3]:
# @title 2. JSON 검수
from google.colab import output
output.enable_custom_widget_manager()
review_coordinates(json_root, review_path, approved_root=approved_root)


In [ ]:
# @title 3. 통과한 JSON만 변형하고 NPZ TAR 저장
IMAGE_SIZE = 128 # @param {type:"integer"}
THICKNESS_PX = 7.0 # @param {type:"number"}
REF_STROKE_THICKNESS_PX = 2.3 # @param {type:"number"}
SCALE_MIN = 0.85 # @param {type:"number"}
SCALE_MAX = 1.15 # @param {type:"number"}
ASPECT_MIN = 0.8 # @param {type:"number"}
ASPECT_MAX = 1.25 # @param {type:"number"}
ROTATION_DEG = 12.0 # @param {type:"number"}
SEED = 42 # @param {type:"integer"}
MAX_TAR_MIB = 256 # @param {type:"integer"}

result = generate_approved(json_root, review_path, destination, approved_root=approved_root,
                           size=IMAGE_SIZE, width=THICKNESS_PX, ref_width=REF_STROKE_THICKNESS_PX,
                           scale_range=(SCALE_MIN, SCALE_MAX), aspect_range=(ASPECT_MIN, ASPECT_MAX),
                           rotation=ROTATION_DEG, seed=SEED, max_bytes=MAX_TAR_MIB * 2**20)
for item in result['stale']:
    print('재검수 필요:', item)
if result['dataset']:
    print(f"통과한 폰트·글자 {result['glyphs']}개, NPZ {result['samples']}장")
    print('학습 DATA_ROOT:', result['dataset'])
else:
    print('현재 통과한 JSON이 없습니다. 2번 셀에서 글자를 검수하세요.')
